In [4]:
# ---------- STEP 1: Import the tools we need ----------
import requests          # lets Python ask a website/API for data
import pandas as pd      # lets us work with data in table form (rows and columns)

# ---------- STEP 2: Decide WHAT data we want ----------
# Left side = World Bank code (the "shelf number" of the dataset)
# Right side = the column name we choose for our own table
INDICATORS = {
    "IT.NET.USER.ZS": "internet_users_pct",     # % of people using the internet
    "GB.XPD.RSDV.GD.ZS": "rd_spend_pct_gdp",    # research & development spending, % of GDP
    "BX.GSR.CCIS.ZS": "ict_exports_pct",        # ICT services as % of service exports
    "BX.GSR.ROYL.CD": "ip_receipts_usd",        # money earned from IP/royalties (US$)
}

# ---------- STEP 3: Decide FOR WHICH countries ----------
# These are 3-letter country codes, separated by ";" as the API needs
COUNTRIES = "IND;DEU;USA;GBR;FRA;BRA;NGA;KEN;ZAF;CHN;JPN;KOR;MEX;IDN;EGY"

# ---------- STEP 4: Fetch each indicator and join them into one table ----------
table = None   # empty at first; we will fill it in the loop

# This loop repeats the same work once for each of the 4 indicators
for code, name in INDICATORS.items():

    # Build the web address of the API request for this indicator
    url = f"https://api.worldbank.org/v2/country/{COUNTRIES}/indicator/{code}"

    # Send the request.
    # "mrv": 5 means "give me the 5 most recent years" (data is often late, so we look back)
    # "per_page": 500 makes sure all rows come back in one go
    r = requests.get(url, params={"format": "json", "mrv": 5, "per_page": 500})

    # If the website returned an error, stop and show it (instead of failing silently)
    r.raise_for_status()

    # The API returns two parts; [1] is the real data. We drop rows with no value (None)
    rows = [x for x in r.json()[1] if x["value"] is not None]

    # Turn those rows into a small table with: country, year of the value, the value
    df = pd.DataFrame([
        {"country": x["country"]["value"],
         f"{name}_year": int(x["date"]),   # which year this number is from
         name: x["value"]}                 # the number itself
        for x in rows
    ])

    # For each country, keep only the LATEST year that has data:
    # sort by year, group by country, then take the last (newest) row
    df = df.sort_values(f"{name}_year").groupby("country").tail(1)

    # First indicator: start the table with it.
    # Other indicators: join them side by side, matching rows by country name
    table = df if table is None else table.merge(df, on="country", how="outer")

# ---------- STEP 5: Save and show ----------
table.to_csv("creative_economy_table.csv", index=False)  # save as a CSV file (index=False skips the row numbers)
print(table)                                             # show the table on screen

             country  internet_users_pct_year  internet_users_pct  \
0             Brazil                     2024           84.463472   
1              China                     2025           91.600000   
2   Egypt, Arab Rep.                     2024           74.648399   
3             France                     2024           88.653816   
4            Germany                     2024           93.499996   
5              India                     2025           70.000000   
6          Indonesia                     2024           72.780755   
7              Japan                     2024           85.541531   
8              Kenya                     2024           34.976225   
9        Korea, Rep.                     2024           97.895638   
10            Mexico                     2024           83.122338   
11           Nigeria                     2024           41.211399   
12      South Africa                     2024           78.357597   
13    United Kingdom              

In [5]:
import numpy as np   # for the log calculation

# ---------- STEP 1: Copy the table so the original stays untouched ----------
data = table.copy()

# ---------- STEP 2: Pick the 4 indicator columns ----------
cols = ["internet_users_pct", "rd_spend_pct_gdp", "ict_exports_pct", "ip_receipts_usd"]

# ---------- STEP 3: Data audit - how many values are missing per column? ----------
print("Missing values per indicator:")
print(data[cols].isna().sum())

# ---------- STEP 4: Take the log of IP receipts ----------
# A few rich countries earn billions, which would squash everyone else near 0.
# log() shrinks big gaps so the scale is fairer.
data["ip_receipts_usd"] = np.log10(data["ip_receipts_usd"])

# ---------- STEP 5: Scale every indicator to 0-100 (min-max) ----------
# The lowest country gets 0, the highest gets 100, others fall in between.
# This lets us compare % with dollars on the same scale.
scores = (data[cols] - data[cols].min()) / (data[cols].max() - data[cols].min()) * 100

# ---------- STEP 6: Average the 4 scores into one index ----------
# skipna=True: if a country is missing one indicator, we average the rest.
# We also record how many indicators were used, so gaps stay visible.
data["index_score"] = scores.mean(axis=1, skipna=True).round(1)
data["indicators_used"] = scores.notna().sum(axis=1)

# ---------- STEP 7: Rank countries and save ----------
data["rank"] = data["index_score"].rank(ascending=False).astype(int)
result = data[["country", "index_score", "rank", "indicators_used"] + cols]
result = result.sort_values("rank")

result = result.rename(columns={"ip_receipts_usd": "ip_receipts_log10"})
result.to_csv("creative_economy_index.csv", index=False)
print(result)

Missing values per indicator:
internet_users_pct    0
rd_spend_pct_gdp      1
ict_exports_pct       0
ip_receipts_usd       1
dtype: int64
             country  index_score  rank  indicators_used  internet_users_pct  \
9        Korea, Rep.         72.5     1                4           97.895638   
14     United States         69.0     2                4           94.693802   
4            Germany         65.1     3                4           93.499996   
13    United Kingdom         62.3     4                4           95.472099   
1              China         61.7     5                4           91.600000   
7              Japan         60.7     6                4           85.541531   
5              India         53.3     7                4           70.000000   
3             France         53.2     8                4           88.653816   
0             Brazil         42.1     9                4           84.463472   
12      South Africa         30.0    10                4     

In [6]:
# Save the final ranked table (with the corrected column name) as a CSV file
result.to_csv("creative_economy_index.csv", index=False)
print("Saved!")

Saved!
